# MediPredict: Hospital Resource Forecasting

## Problem Statement
Small hospitals generate large amounts of patient data but rarely analyze it to predict resource needs like beds, oxygen, and critical care.

## Team Size
3 members


## Data Science Lifecycle
1. Question
2. Data
3. Cleaning
4. Analysis
5. Modeling
6. Insight


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

## Data Loading

In [ ]:
raw_df = pd.read_csv('../data/raw/hospital_data.csv')
clean_df = pd.read_csv('../data/processed/cleaned_hospital_data.csv')

raw_df.head()

## Data Cleaning

In [ ]:
clean_df['date'] = pd.to_datetime(clean_df['date'])
clean_df.info()

## Missing Value Handling

In [ ]:
raw_df.isnull().sum()

## Duplicate Handling

In [ ]:
raw_df.duplicated().sum(), clean_df.duplicated().sum()

## Summary Statistics

In [ ]:
clean_df.describe(include='all').T

## Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(clean_df['date'], clean_df['total_patients'])
ax.set_title('Patient Trend Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Total Patients')
plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(clean_df['date'], clean_df['admitted_patients'], color='tab:orange')
ax.set_title('Admitted Patients Trend')
ax.set_xlabel('Date')
ax.set_ylabel('Admitted Patients')
plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(clean_df['date'], clean_df['oxygen_patients'], color='tab:green')
ax.set_title('Oxygen Demand Trend')
ax.set_xlabel('Date')
ax.set_ylabel('Oxygen Patients')
plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(clean_df['date'], clean_df['icu_patients'], color='tab:red')
ax.set_title('ICU Demand Trend')
ax.set_xlabel('Date')
ax.set_ylabel('ICU Patients')
plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(clean_df['total_patients'], bins=15, color='skyblue', edgecolor='black')
ax.set_title('Distribution of Total Patients')
ax.set_xlabel('Total Patients')
ax.set_ylabel('Frequency')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(clean_df['emergency_patients'], clean_df['oxygen_patients'], alpha=0.7, color='purple')
ax.set_title('Emergency vs Oxygen Demand')
ax.set_xlabel('Emergency Patients')
ax.set_ylabel('Oxygen Patients')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.boxplot([clean_df['total_patients'], clean_df['admitted_patients'], clean_df['oxygen_patients']], labels=['Total','Admitted','Oxygen'])
ax.set_title('Boxplot for Outlier Detection')
plt.show()

## Model Training

In [ ]:
model_df = clean_df.sort_values('date').copy()
model_df['next_day_bed_requirement'] = model_df['admitted_patients'].shift(-1)
model_df['next_day_oxygen_requirement'] = model_df['oxygen_patients'].shift(-1)
model_df['next_day_icu_requirement'] = model_df['icu_patients'].shift(-1)
model_df = model_df.dropna()

features = [
    'total_patients','emergency_patients','admitted_patients','icu_patients','oxygen_patients',
    'discharged_patients','available_beds','available_oxygen_cylinders','available_icu_beds',
    'avg_patient_age','high_risk_patients'
]

def evaluate_models(target_col):
    X = model_df[features]
    y = model_df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    lr = LinearRegression()
    lr.fit(X_train, y_train)
    lr_pred = lr.predict(X_test)

    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    results = {
        'LinearRegression': {
            'MAE': mean_absolute_error(y_test, lr_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, lr_pred)),
            'R2': r2_score(y_test, lr_pred)
        },
        'RandomForestRegressor': {
            'MAE': mean_absolute_error(y_test, rf_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, rf_pred)),
            'R2': r2_score(y_test, rf_pred)
        }
    }
    return results

bed_results = evaluate_models('next_day_bed_requirement')
oxygen_results = evaluate_models('next_day_oxygen_requirement')
icu_results = evaluate_models('next_day_icu_requirement')

bed_results, oxygen_results, icu_results

## Final Insights
- Patient load trends are useful early indicators of resource demand.
- Random Forest often performs better in non-linear patterns.
- Small hospitals can use simple forecasting to improve readiness.


## Assumptions and Limitations
### Assumptions
- The synthetic data pattern is close to real trends.
- Next-day demand is affected by current day statistics.

### Limitations
- Data is synthetic and simplified.
- No external factors (policy change, disasters) are modeled.
- This is a beginner-level baseline model.
